# NYC Taxi Times

**Author:** Clarice Mottet  

---

## Objective

Build a model to predict the total ride duration for NYC taxis

---

## Dataset
**Source:**
- kaggle


## Table of Contents

- [1. Load Data](#1-load-data)
- [2. Data Cleaning](#2-data-cleaning)
- [3. Exploratory Data Analysis](#3-exploratory-data-analysis)
- [4. Feature Engineering](#4-feature-engineering)
- [5. Modeling](#5-modeling)
- [6. Model Evaluation](#6-model-evaluation)


## Notes

- Random seed = 42
- Train/Test split = pre-determined by datasets given
- Evaluation metric = Root Mean Squared Log Error

# 1. Load Data

In [117]:
#libraries

import pandas as pd
import numpy as np
import os
from pathlib import Path
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans


project_root = Path.cwd().parent          # if notebook is inside notebooks/


## Import in train and test data

In [61]:
#import in the datasets
#quick look over

#establish  file path

raw_path = project_root / "data" / "raw"
os.listdir(raw_path)

df_test = pd.read_csv(raw_path / "test.csv", dtype = str)
df_train = pd.read_csv(raw_path / "train.csv", dtype = str)
df_sample = pd.read_csv(raw_path / "sample_submission.csv", dtype = str)

* Column Check:
train dataset has the columns "dropoff_datetime" and "trip_duration" which test does not

In [62]:
#perform columns check

# print(df_test.columns)
# print(df_train.columns)

list_test_col = list(df_test.columns)
list_train_col = list(df_train.columns)

list_all_col = list_test_col + list_train_col
list_all_col = list(set(list_all_col))
print(list_all_col)

list_test_doesnt = [x for x in list_all_col if x not in list_test_col]
print(list_test_doesnt)

list_train_doesnt = [x for x in list_all_col if x not in list_train_col]
# print(list_train_doesnt)

#so train has the resulting columns for me to be able to compare and train to, theres dropoff_datetime and trip duration



['id', 'pickup_latitude', 'vendor_id', 'passenger_count', 'dropoff_latitude', 'trip_duration', 'pickup_longitude', 'store_and_fwd_flag', 'pickup_datetime', 'dropoff_datetime', 'dropoff_longitude']
['trip_duration', 'dropoff_datetime']


# 2. Data Cleaning

* dtype
* nulls
* duplicates

* Data Type check:

In [63]:
# print(df_train.head())
# print(df_train.columns)

#need to identify dtypes
for col in df_train.columns:
    df_train[col] = df_train[col].str.strip()

#numeric columns
list_col_num = ['vendor_id', 'passenger_count', 'pickup_longitude', 'pickup_latitude', 'dropoff_longitude', 'dropoff_latitude',
                'trip_duration']
list_col_date = ['pickup_datetime', 'dropoff_datetime']


* convert train and test to numeric

In [64]:
df_ = df_train.copy()

list_convert_num = [x for x in list_col_num if x in df_.columns]
df_[list_convert_num] = df_[list_convert_num].apply(pd.to_numeric)
df_train = df_.copy()

In [65]:
df_t = df_test.copy()

list_convert_num = [x for x in list_col_num if x in df_t.columns]
df_t[list_convert_num] = df_t[list_convert_num].apply(pd.to_numeric)
df_test = df_t.copy()

* convert train and test to datetime

In [66]:
df_ = df_train.copy()

list_convert_date = [x for x in list_col_date if x in df_.columns]
df_[list_convert_date] = df_[list_convert_date].apply(pd.to_datetime)
df_train = df_.copy()

In [68]:
df_ = df_test.copy()

list_convert_date = [x for x in list_col_date if x in df_.columns]
df_[list_convert_date] = df_[list_convert_date].apply(pd.to_datetime)


In [69]:
df_.dtypes

id                            object
vendor_id                      int64
pickup_datetime       datetime64[ns]
passenger_count                int64
pickup_longitude             float64
pickup_latitude              float64
dropoff_longitude            float64
dropoff_latitude             float64
store_and_fwd_flag            object
dtype: object

* there are no nulls

In [70]:
df_train.isnull().sum()

id                    0
vendor_id             0
pickup_datetime       0
dropoff_datetime      0
passenger_count       0
pickup_longitude      0
pickup_latitude       0
dropoff_longitude     0
dropoff_latitude      0
store_and_fwd_flag    0
trip_duration         0
dtype: int64

In [18]:
df_train.isna().sum()

id                    0
vendor_id             0
pickup_datetime       0
dropoff_datetime      0
passenger_count       0
pickup_longitude      0
pickup_latitude       0
dropoff_longitude     0
dropoff_latitude      0
store_and_fwd_flag    0
trip_duration         0
dtype: int64

In [71]:
null_counts = df_train.isna().sum()
null_counts = null_counts[null_counts > 0]
print(null_counts)

Series([], dtype: int64)


In [72]:
null_counts = df_test.isna().sum()
null_counts = null_counts[null_counts > 0]
print(null_counts)

Series([], dtype: int64)


In [74]:
#target variables
var_target = ['dropoff_datetime','trip_duration']

# df_test.columns

* check to make sure trip duration is calculated properly, it is 

In [75]:
df_view = df_train.copy()
df_view['dur_calc'] = (df_view['dropoff_datetime'] - df_view['pickup_datetime']).dt.total_seconds().astype(int)

df_view.head()

df_view['check_dur'] = (df_view['trip_duration'] - df_view['dur_calc']).abs()
df_view_ = df_view[df_view['check_dur']!=0]
# df_view_.head()
df_test.columns

Index(['id', 'vendor_id', 'pickup_datetime', 'passenger_count',
       'pickup_longitude', 'pickup_latitude', 'dropoff_longitude',
       'dropoff_latitude', 'store_and_fwd_flag'],
      dtype='object')

* there are no duplicates by the ID variabe in both train and test

* there are dupliactes by all other fields (1 in test, 7 in train)

In [76]:
#check for duplicates

df_view = df_test.copy()
df_view.drop_duplicates(subset = ['pickup_datetime', 'passenger_count',
       'pickup_longitude', 'pickup_latitude', 'dropoff_longitude',
       'dropoff_latitude', 'store_and_fwd_flag'], inplace = True)



In [77]:
df_view = df_train.copy()
df_view.drop_duplicates(subset = ['pickup_datetime', 'passenger_count',
       'pickup_longitude', 'pickup_latitude', 'dropoff_longitude',
       'dropoff_latitude', 'store_and_fwd_flag'], inplace = True)



* check for invalid values

In [78]:
df_train.describe()

,vendor_id,pickup_datetime,dropoff_datetime,passenger_count,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,trip_duration
count,1.458644e+06,1458644,1458644,1.458644e+06,1.458644e+06,1.458644e+06,1.458644e+06,1.458644e+06,1.458644e+06
mean,1.534950e+00,2016-04-01 10:10:24.940037120,2016-04-01 10:26:24.432310528,1.664530e+00,-7.397349e+01,4.075092e+01,-7.397342e+01,4.075180e+01,9.594923e+02
min,1.000000e+00,2016-01-01 00:00:17,2016-01-01 00:03:31,0.000000e+00,-1.219333e+02,3.435970e+01,-1.219333e+02,3.218114e+01,1.000000e+00
25%,1.000000e+00,2016-02-17 16:46:04.249999872,2016-02-17 17:05:32.500000,1.000000e+00,-7.399187e+01,4.073735e+01,-7.399133e+01,4.073588e+01,3.970000e+02
50%,2.000000e+00,2016-04-01 17:19:40,2016-04-01 17:35:12,1.000000e+00,-7.398174e+01,4.075410e+01,-7.397975e+01,4.075452e+01,6.620000e+02
75%,2.000000e+00,2016-05-15 03:56:08.750000128,2016-05-15 04:10:51.750000128,2.000000e+00,-7.396733e+01,4.076836e+01,-7.396301e+01,4.076981e+01,1.075000e+03
max,2.000000e+00,2016-06-30 23:59:39,2016-07-01 23:02:03,9.000000e+00,-6.133553e+01,5.188108e+01,-6.133553e+01,4.392103e+01,3.526282e+06
std,4.987772e-01,NaN,NaN,1.314242e+00,7.090186e-02,3.288119e-02,7.064327e-02,3.589056e-02,5.237432e+03


# 3. Exploratory Data Analysis

* variable exploration
* distribution

1. Target Variable Analysis
   • histogram
   • log histogram
   • boxplot

2. Datetime Analysis
   • hour
   • weekday
   • month

3. Geographic Analysis
   • pickup map
   • dropoff map
   • distance distributions

4. Numeric Feature Analysis
   • passenger_count
   • haversine_distance
   • manhattan_distance

5. Categorical Feature Analysis
   • vendor
   • store_and_fwd_flag

6. Relationships with Target
   • scatterplots
   • boxplots


## Train
* passenger_count - no issues
* pickup_datetime:
    - only jan 2016 - july 2016
    - by date: uniform
    - by time of day: fewer in early morning, some local peaks
    - by day of week: pretty uniform
* passenger_counts:
    - approx 2/3 are 1 passenger only
    - next most as 2 passengers and then 3-6 are about the same frequency
* longitude and latitude:
    - determined how many clusters are needed
    - there are approx 6k instances where the pickup point is the same as the dropoff point
    

In [145]:
df_train.describe()

,vendor_id,pickup_datetime,dropoff_datetime,passenger_count,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,trip_duration,hour,hour_sin,hour_cos,day_of_week,manhattan_distance
count,1.458644e+06,1458644,1458644,1.458644e+06,1.458644e+06,1.458644e+06,1.458644e+06,1.458644e+06,1.458644e+06,1.458644e+06,1.458644e+06,1.458644e+06,1.458644e+06,1.458644e+06
mean,1.534950e+00,2016-04-01 10:10:24.940037632,2016-04-01 10:26:24.432311296,1.664530e+00,-7.397349e+01,4.075092e+01,-7.397342e+01,4.075180e+01,9.594923e+02,1.360648e+01,-2.017288e-01,-8.447460e-02,3.050375e+00,4.446650e+00
min,1.000000e+00,2016-01-01 00:00:17,2016-01-01 00:03:31,0.000000e+00,-1.219333e+02,3.435970e+01,-1.219333e+02,3.218114e+01,1.000000e+00,0.000000e+00,-1.000000e+00,-1.000000e+00,0.000000e+00,0.000000e+00
25%,1.000000e+00,2016-02-17 16:46:04.249999872,2016-02-17 17:05:32.500000,1.000000e+00,-7.399187e+01,4.073735e+01,-7.399133e+01,4.073588e+01,3.970000e+02,9.000000e+00,-8.660254e-01,-7.071068e-01,1.000000e+00,1.570939e+00
50%,2.000000e+00,2016-04-01 17:19:40,2016-04-01 17:35:12,1.000000e+00,-7.398174e+01,4.075410e+01,-7.397975e+01,4.075452e+01,6.620000e+02,1.400000e+01,-2.588190e-01,-2.588190e-01,3.000000e+00,2.688802e+00
75%,2.000000e+00,2016-05-15 03:56:08.750000128,2016-05-15 04:10:51.750000128,2.000000e+00,-7.396733e+01,4.076836e+01,-7.396301e+01,4.076981e+01,1.075000e+03,1.900000e+01,5.000000e-01,7.071068e-01,5.000000e+00,4.998980e+00
max,2.000000e+00,2016-06-30 23:59:39,2016-07-01 23:02:03,9.000000e+00,-6.133553e+01,5.188108e+01,-6.133553e+01,4.392103e+01,3.526282e+06,2.300000e+01,1.000000e+00,1.000000e+00,6.000000e+00,1.336846e+03
std,4.987772e-01,NaN,NaN,1.314242e+00,7.090186e-02,3.288119e-02,7.064327e-02,3.589056e-02,5.237432e+03,6.399693e+00,6.728468e-01,7.067159e-01,1.954039e+00,5.659821e+00
